# 01_02 — Master Calendar + CEX Diagnostics (Report)

Reading-side companion to `run_data_prep.py`. The script merges the
build cells of NB01 (master calendar from Bybit klines / OI / funding
bounds) and NB02 (Bybit + Binance left-join + spread diagnostics) into
a single linear pipeline (calendar → diagnostics) and dumps five
artefacts:

- `data/analysis/windows/master_calendar_1h.parquet` — 1-col, 41 328-row hourly grid
- `data/analysis/windows/window_metadata.json` — full / core / series_bounds
- `data/analysis/reports/calendar_qa.json` — calendar audit
- `data/analysis/datasets/cex_diagnostics_1h.parquet` — 12-col diagnostic panel
- `data/analysis/reports/diagnostics_qa.json` — spread / correlation audit

This notebook reads those files and verifies the artefacts against the
downstream contract without re-running any computation. This report
supersedes the original exploratory notebooks (`01_calendar.ipynb`,
`02_diagnostics.ipynb`), which are not part of the replication package.

**Contract with NB03 / `run_core_panel.py`.** `master_calendar_1h.parquet`
(1 col × 41 328 rows) and `window_metadata.json` (4 top-level keys)
schemas are locked. The 3 audit / diagnostic artefacts have no downstream
consumer but are preserved bit-for-bit for legacy parity. Any drift on
the calendar or window metadata breaks NB03 → NB04 → NB07 / NB08
/ NB09 + the factorised scripts.

## 0. Setup & auto-execution

In [ ]:
# ── Setup ──
import sys, json, subprocess, time
from pathlib import Path
sys.path.insert(0, "..")
from config import CFG, REPORTS_DIR, WINDOWS_DIR, DATASETS_DIR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style: serif, print-friendly, monochrome (cf. 03/04/08 reports).
plt.rcParams.update({
    "figure.dpi":      150,
    "savefig.dpi":     300,
    "font.family":     "serif",
    "font.size":       11,
    "axes.titlesize":  12,
    "axes.labelsize":  11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.grid":       True,
    "grid.alpha":      0.3,
    "grid.linewidth":  0.5,
})

# Toggles
FORCE_RERUN = False    # if True, regenerate all outputs via the script

SCRIPT  = Path.cwd().parent / "scripts" / "run_data_prep.py"
OUTPUTS = {
    "master_calendar": CFG.FILES.master_calendar,
    "window_metadata": CFG.FILES.window_metadata,
    "calendar_qa":     REPORTS_DIR / "calendar_qa.json",
    "cex_diagnostics": CFG.FILES.cex_diagnostics,
    "diagnostics_qa":  REPORTS_DIR / "diagnostics_qa.json",
}

print("Setup OK")
print(f"  FORCE_RERUN = {FORCE_RERUN}")

In [ ]:
# ── Auto-execute if outputs are missing or FORCE_RERUN is set ──
missing = [t for t, p in OUTPUTS.items() if not p.exists()]

if FORCE_RERUN:
    print("FORCE_RERUN=True → re-running run_data_prep.py...", flush=True)
    run = True
elif missing:
    print(f"Outputs missing ({missing}) → running run_data_prep.py...", flush=True)
    run = True
else:
    print("All outputs present, skipping recompute (FORCE_RERUN=False)")
    for t, p in OUTPUTS.items():
        print(f"  [{t}] {p.name}")
    run = False

if run:
    assert SCRIPT.exists(), SCRIPT
    cmd = [sys.executable, str(SCRIPT)]
    print("Running:", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=str(Path.cwd().parent), check=True)
    print(f"\nScript done — {time.time()-t0:.1f}s")

In [ ]:
# ── Load all 5 artefacts ──
df_cal = pd.read_parquet(OUTPUTS["master_calendar"], engine="pyarrow")
df_cal["date"] = pd.to_datetime(df_cal["date"], utc=True)

with open(OUTPUTS["window_metadata"]) as f:
    meta = json.load(f)
with open(OUTPUTS["calendar_qa"]) as f:
    qa_cal = json.load(f)

df_cex = pd.read_parquet(OUTPUTS["cex_diagnostics"], engine="pyarrow")
df_cex["date"] = pd.to_datetime(df_cex["date"], utc=True)

with open(OUTPUTS["diagnostics_qa"]) as f:
    qa_diag = json.load(f)

print(f"calendar:        {len(df_cal):,} rows × {df_cal.shape[1]} cols")
print(f"window_metadata: {list(meta.keys())}")
print(f"calendar_qa:     status = {qa_cal['status']}")
print(f"cex_diagnostics: {len(df_cex):,} rows × {df_cex.shape[1]} cols")
print(f"diagnostics_qa:  status = {qa_diag['status']}")

## 1. Master calendar

Doit afficher 41 328 lignes, monotone, espacement uniforme 1h, fuseau
UTC, sur la fenêtre `[2021-03-15, 2025-12-01)`. Les asserts ci-dessous
rejouent les contrôles QA de NB01 sur le parquet relu.

In [ ]:
print("═══ Master calendar ═══\n")
print(f"  Rows:        {len(df_cal):,}")
print(f"  Cols:        {df_cal.shape[1]}  ({list(df_cal.columns)})")
print(f"  Date range:  [{df_cal['date'].min()}, {df_cal['date'].max()}]")
print(f"  Frequency:   {CFG.FREQ}")

# Re-asserts (mirror NB01 cell 8b739e4b)
diffs = df_cal["date"].diff().dropna()
checks = [
    ("row count == 41 328",  len(df_cal) == 41328),
    ("monotonic increasing", bool(df_cal["date"].is_monotonic_increasing)),
    ("uniform 1h spacing",   bool(diffs.eq(pd.Timedelta(hours=1)).all())),
    ("timezone == UTC",      str(df_cal["date"].dt.tz) == "UTC"),
]
print("\n  QA replays:")
for label, ok in checks:
    flag = "OK" if ok else "FAIL"
    print(f"    [{flag}] {label}")
    assert ok, label

print(f"\n  calendar_qa.json: {qa_cal}")

## 2. Window metadata

Convention `[start, end_excl)`, timestamp = bucket-start. `series_bounds`
garde les bornes hautes brutes (avant `+1h`) — ce sont les `bucket_start`
du dernier bucket de chaque série, pas un `end_excl`. En pratique les 3
séries Bybit ont des bornes identiques → `core_window ≡ full_window`.

In [ ]:
print("═══ Window metadata ═══\n")
print("── Full window ──")
for k, v in meta["full_window"].items():
    print(f"  {k:10s}: {v}")
print("\n── Core window ──")
for k, v in meta["core_window"].items():
    print(f"  {k:10s}: {v}")
print("\n── Series bounds (pre +1h) ──")
for name, b in meta["series_bounds"].items():
    print(f"  {name:14s}: [{b['start']}, {b['end']}]")
print(f"\n── Convention ──\n  {meta['convention']}")

## 3. CEX diagnostics — panel

Panel 12 colonnes × 41 328 lignes. Les 4 colonnes-clé
(`close_bybit`, `close_binance`, `oi_bybit`, `funding_bybit`) doivent
être à 0 % de manquants. `ret_bybit` / `ret_binance` ont 1 NaN à `t=0`
par construction (`np.log(...).diff()`).

In [ ]:
print("═══ CEX diagnostics — panel ═══\n")
print(f"  Shape: {df_cex.shape}")
print(f"  Cols : {list(df_cex.columns)}")

miss = (df_cex.isna().sum() / len(df_cex) * 100).round(4)
miss = miss.sort_values(ascending=False).rename("missing_pct")
print("\n── Missing rates (%) ──")
print(miss.to_string())

## 4. Spread analysis

Reproduit le bloc de NB02 cell `344eea2a` à partir du panel relu.
Note méthodologique : le seuil d'affichage (0.999) et le seuil QA
(0.99) sont volontairement différents — cosmétique-vs-contrat,
préservé verbatim depuis NB02. Avec `corr ≈ 0.99895` la ligne
d'affichage imprime « Check… » alors que `qa_diag['status']` vaut
`PASS`. Aucun impact sur les artefacts.

In [ ]:
spread = df_cex["price_spread_bps"].dropna()

print("═" * 55)
print("PRICE SPREAD: BYBIT vs BINANCE (bps)")
print("═" * 55)
print(f"Mean (bias)        : {spread.mean():.4f} bps")
print(f"Std (volatility)   : {spread.std():.4f} bps")
print(f"Mean |spread|      : {spread.abs().mean():.4f} bps")
print(f"Median |spread|    : {spread.abs().median():.4f} bps")
print(f"P99 |spread|       : {np.percentile(spread.abs(), 99):.2f} bps")
print(f"P99.9 |spread|     : {np.percentile(spread.abs(), 99.9):.2f} bps")

corr = df_cex[["ret_bybit", "ret_binance"]].dropna().corr().iloc[0, 1]
print(f"\nReturn correlation : {corr:.6f}")
print("→ Confirms near-perfect market integration"
      if corr > 0.999 else "→ Check for micro-structure differences")

print(f"\n── diagnostics_qa.json ──")
for k, v in qa_diag.items():
    print(f"  {k:24s}: {v}")

## 5. Sanity check graphique — `price_spread_bps` au cours du temps

Le spread Bybit-Binance est le canari du merge cross-venue. Doit
osciller autour de 0 avec des spikes ponctuels lors des épisodes de
stress. Toute dérive systématique ou trou prolongé suggère un
désalignement temporel entre les deux venues (fuseau, horodatage,
fenêtre).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.plot(df_cex["date"], df_cex["price_spread_bps"],
        color="black", linewidth=0.5)
ax.axhline(0, color="black", linestyle=":", linewidth=0.6)
ax.set_xlabel("")
ax.set_ylabel("price_spread_bps")
ax.margins(x=0)
plt.tight_layout()
plt.show()

## Next step

`master_calendar_1h.parquet` et `window_metadata.json` sont les deux
artefacts de cette chaîne consommés en aval. Ils alimentent :
- **NB03** / `run_core_panel.py` (panel pré-DeFi) — qui lit le
  calendrier comme grille horaire de référence et `window_metadata`
  pour le slice de la *core window* dans l'audit des manquants.

Les 3 autres artefacts (`calendar_qa.json`, `cex_diagnostics_1h.parquet`,
`diagnostics_qa.json`) sont des terminaux de diagnostic / audit, sans
consommateur aval. Toute régression sur le calendrier ou la métadonnée
de fenêtre est immédiatement visible en aval (NB03 → NB04 →
NB07 / NB08 / NB09).